# QuantumTX AH — Deep EDA Notebook

Comprehensive exploratory analysis of the 2024 Alexandra Hospital dataset.
Covers responder profiling, dropout analysis, dosage-response, comorbidity
patterns, and feature correlations.

**Run top-to-bottom.** All artefacts (PNGs, CSVs) are saved to `reports/eda_artefacts/`.

---

## Section 0 — Setup & Data Load

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend — required for save_fig
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import Markdown, display

# Project root — works whether kernel CWD is project root or notebooks/
PROJECT_ROOT = Path(".").resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

ARTEFACTS = PROJECT_ROOT / "reports" / "eda_artefacts"
ARTEFACTS.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
PALETTE = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2","#937860","#DA8BC3","#8C8C8C"]

def save_fig(name: str):
    """Save current matplotlib figure to artefacts dir and display inline."""
    plt.savefig(ARTEFACTS / name, dpi=150, bbox_inches="tight")
    print(f"Saved: {ARTEFACTS / name}")
    plt.show()
    plt.close("all")

print("Setup complete.")


In [ ]:
df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "featured.parquet")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Derived subsets used throughout
df_followup      = df[df["has_followup"] == "Y"].copy()
df_dropout_known = df[df["is_dropout"].notna()].copy()

# Column groups
HAS_FLAGS   = sorted([c for c in df.columns if c.startswith("has_") and c != "has_followup"])
GRP_FLAGS   = sorted([c for c in df.columns if c.startswith("grp_")])
RGN_FLAGS   = sorted([c for c in df.columns if c.startswith("rgn_")])
IMPROV_COLS = [c for c in ["vas_improvement","tug_improvement","sst_improvement",
                             "normal_gs_improvement","fast_gs_improvement","sppb_improvement"]
               if c in df.columns]
_ALL_IMPROV_LABELS = {
    "vas_improvement":       "VAS Pain",
    "tug_improvement":       "TUG",
    "sst_improvement":       "5xSST",
    "normal_gs_improvement": "Normal GS",
    "fast_gs_improvement":   "Fast GS",
    "sppb_improvement":      "SPPB",
}
IMPROV_LABELS = {k: v for k, v in _ALL_IMPROV_LABELS.items() if k in IMPROV_COLS}
# MCID thresholds. None = percentage-based (computed per-row in analysis cells).
MCID_THRESHOLDS = {
    "vas_improvement":       2.0,    # ≥2 point reduction in VAS (0-10 scale)
    "tug_improvement":       3.0,    # ≥3 s improvement in TUG
    "sst_improvement":       None,   # ≥10% of pre_5xsst_s (relative — see Section 4)
    "normal_gs_improvement": 0.05,   # ≥0.05 m/s improvement
    "fast_gs_improvement":   0.10,   # ≥0.10 m/s improvement
    "sppb_improvement":      1.0,    # ≥1 point improvement in SPPB
}
NUMERIC_FEATS = [c for c in ["age","baseline_sppb","pre_normal_gs_ms","pre_tug_s",
                               "pre_5xsst_s","pre_vas","pre_fast_gs_ms",
                               "n_flags","n_regions","n_groups"] if c in df.columns]

# Reproduce README overview stats
total_n      = len(df)
n_followup   = (df["has_followup"] == "Y").sum()
n_responders = int((df_followup["overall_responder"] == 1).sum())
print(f"\nOverview:")
print(f"  Total patients : {total_n:,}")
print(f"  With follow-up : {n_followup:,}  ({n_followup/total_n*100:.1f}%)")
print(f"  Responders     : {n_responders:,}  ({n_responders/n_followup*100:.1f}% of follow-up)")
print(f"  Dropouts       : {int(df['is_dropout'].sum()):,}")


---
## Section 1 — Cohort Profiles

How are patients distributed across cohorts? We profile each cohort by size,
age, gender, follow-up rate, and responder rate to establish baseline context
for all later analyses.

In [ ]:
# Build usage frequency short labels for column names
USAGE_MAP = {
    "Once (1x/week, one leg)":               "pct_1x",
    "Twice (2x/week, one leg per session)":  "pct_2x",
    "L+R 10 (20-min session, 10 min each leg)": "pct_lr",
}

rows = []
for cohort in sorted(df["cohort"].dropna().unique()):
    sub    = df[df["cohort"] == cohort]
    sub_fu = sub[sub["has_followup"] == "Y"]
    n_sub  = len(sub)
    row = {
        "cohort":            cohort,
        "n":                 n_sub,
        "pct_of_total":      round(n_sub / len(df) * 100, 1) if len(df) else float("nan"),
        "mean_age":          round(sub["age"].mean(), 1),
        "std_age":           round(sub["age"].std(), 1),
        "pct_female":        round((sub["gender"] == "F").sum() / n_sub * 100, 1) if n_sub else float("nan"),
        "n_followup":        len(sub_fu),
        "pct_followup":      round(len(sub_fu) / n_sub * 100, 1) if n_sub else float("nan"),
        "pct_responder":     round((sub_fu["overall_responder"] == 1).sum() / len(sub_fu) * 100, 1)
                             if len(sub_fu) >= 5 else float("nan"),  # suppress rate for very small sub-cohorts
    }
    # Usage frequency breakdown per cohort
    for freq_val, col_name in USAGE_MAP.items():
        row[col_name] = round((sub["usage_frequency"] == freq_val).sum() / n_sub * 100, 1) if n_sub else float("nan")
    rows.append(row)

cohort_df = pd.DataFrame(rows)
cohort_df.to_csv(ARTEFACTS / "cohort_profiles.csv", index=False)
display(cohort_df)


In [ ]:
cohort_colors = [PALETTE[i % len(PALETTE)] for i in range(len(cohort_df))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(cohort_df["cohort"], cohort_df["n"], color=cohort_colors)
axes[0].set_title("Patients per Cohort")
axes[0].set_ylabel("N")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(cohort_df["cohort"], cohort_df["pct_followup"], color=cohort_colors)
axes[1].axhline(n_followup / total_n * 100, color="red", linestyle="--", label="Overall avg")
axes[1].set_title("Follow-up Rate by Cohort")
axes[1].set_ylabel("% with Follow-up")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()

plt.tight_layout()
save_fig("cohort_overview.png")


**Note:** ~60% of patients are classified as Unclassified (no comorbidity tags assigned). Subgroup analyses within named cohorts (Frailty, Neurological, etc.) are based on smaller samples and should be interpreted with caution.

---
## Section 2 — Responder Rates by Subgroup

Which patient groups respond best? We compute the percentage of overall_responder
(among follow-up patients) for every meaningful subgroup: cohort, age band, gender,
usage frequency, and each comorbidity flag.

In [ ]:
def responder_rate(subdf):
    """% overall_responder among follow-up patients in subdf. Returns (rate, n_followup)."""
    fu = subdf[subdf["has_followup"] == "Y"]
    if len(fu) < 5:
        return float("nan"), len(fu)
    return round((fu["overall_responder"] == 1).sum() / len(fu) * 100, 1), len(fu)

rows = []
for val in sorted(df["cohort"].dropna().unique()):
    r, n = responder_rate(df[df["cohort"] == val])
    rows.append({"group": "cohort", "value": val, "n_followup": n, "responder_rate_pct": r})

for val in sorted(df["age_band"].dropna().unique()):
    r, n = responder_rate(df[df["age_band"] == val])
    rows.append({"group": "age_band", "value": str(val), "n_followup": n, "responder_rate_pct": r})

for val in [v for v in df["gender"].dropna().unique() if v != "__missing__"]:
    r, n = responder_rate(df[df["gender"] == val])
    rows.append({"group": "gender", "value": val, "n_followup": n, "responder_rate_pct": r})

for val in sorted(df["usage_frequency"].dropna().unique()):
    if val == "__missing__":
        continue
    r, n = responder_rate(df[df["usage_frequency"] == val])
    rows.append({"group": "usage_frequency", "value": str(val), "n_followup": n, "responder_rate_pct": r})

for flag in HAS_FLAGS:
    r, n = responder_rate(df[df[flag] == 1])
    rows.append({"group": "comorbidity_flag", "value": flag, "n_followup": n, "responder_rate_pct": r})

subgroup_df = pd.DataFrame(rows)
suppressed = subgroup_df[subgroup_df["responder_rate_pct"].isna()]
if len(suppressed):
    print(f"Suppressed {len(suppressed)} subgroups with n_followup < 5: {suppressed['value'].tolist()}")
subgroup_df = subgroup_df.dropna(subset=["responder_rate_pct"])
subgroup_df.to_csv(ARTEFACTS / "responder_rates_by_subgroup.csv", index=False)
display(subgroup_df.sort_values("responder_rate_pct", ascending=False).head(20))


In [ ]:
flag_df = (subgroup_df[subgroup_df["group"] == "comorbidity_flag"]
           .sort_values("responder_rate_pct", ascending=True))
overall_avg = (df_followup["overall_responder"] == 1).sum() / len(df_followup) * 100

fig, ax = plt.subplots(figsize=(8, max(5, len(flag_df) * 0.38)))
colors = ["#C44E52" if r < 50 else "#4C72B0" if r > 80 else "#DD8452"
          for r in flag_df["responder_rate_pct"]]
ax.barh(flag_df["value"].str.replace("has_", ""), flag_df["responder_rate_pct"], color=colors)
ax.axvline(overall_avg, color="black", linestyle="--", label=f"Overall avg ({overall_avg:.0f}%)")
ax.axvline(50, color="#C44E52", linestyle=":", alpha=0.6, label="50% threshold (low)")
ax.axvline(80, color="#4C72B0", linestyle=":", alpha=0.6, label="80% threshold (high)")
ax.set_xlabel("Responder Rate (%)")
ax.set_title("Responder Rate by Comorbidity Flag\n(red = <50%, orange = mid, blue = >80%)")
ax.legend(fontsize=8)
plt.tight_layout()
save_fig("responder_rates_by_flag.png")


---
## Section 3 — Responder vs Non-Responder Profiles

We compare baseline characteristics between responders and non-responders
using Mann-Whitney U tests (non-parametric, appropriate for skewed clinical data).

In [ ]:
resp     = df_followup[df_followup["overall_responder"] == 1]
non_resp = df_followup[df_followup["overall_responder"] == 0]
print(f"Responders: {len(resp)}, Non-responders: {len(non_resp)}")

# Spec-defined feature list (Section 3 analysis only)
S3_FEATS = [c for c in ["age", "baseline_sppb", "pre_normal_gs_ms", "pre_tug_s",
                          "pre_5xsst_s", "pre_vas", "pre_fast_gs_ms", "n_flags"]
             if c in df_followup.columns]

rows = []
for feat in S3_FEATS:
    a = resp[feat].dropna()
    b = non_resp[feat].dropna()
    if len(a) < 5 or len(b) < 5:
        continue
    _, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    rows.append({
        "feature":            feat,
        "responder_mean":     round(a.mean(), 2),
        "responder_std":      round(a.std(), 2),
        "non_responder_mean": round(b.mean(), 2),
        "non_responder_std":  round(b.std(), 2),
        "p_value":            round(p, 4),
        "significant_p05":    p < 0.05,
    })

comp_df = pd.DataFrame(rows).sort_values("p_value")
comp_df.to_csv(ARTEFACTS / "responder_profile_comparison.csv", index=False)
display(comp_df)


In [ ]:
top_feats = comp_df.head(4)["feature"].tolist()
n_top = len(top_feats)
if n_top == 0:
    print("No features with sufficient data for violin plots.")
else:
    fig, axes = plt.subplots(1, n_top, figsize=(n_top * 4, 5))
    if n_top == 1:
        axes = [axes]
    for ax, feat in zip(axes, top_feats):
        data = df_followup[[feat, "overall_responder"]].dropna().copy()
        data["Responder"] = data["overall_responder"].map({1: "Yes", 0: "No"})
        sns.violinplot(data=data, x="Responder", y=feat, ax=ax,
                       palette={"Yes": "#4C72B0", "No": "#DD8452"}, inner="box")
        p_val = comp_df.loc[comp_df["feature"] == feat, "p_value"].values
        title = feat.replace("_", " ").title()
        if len(p_val):
            title += f"\np={p_val[0]:.4f}"
        ax.set_title(title)
        ax.set_xlabel("")
    plt.suptitle("Top Differentiating Features: Responders vs Non-Responders", fontsize=12, y=1.02)
    plt.tight_layout()
    save_fig("responder_profiles.png")


---
## Section 4 — Individual Test Responder Rates

`overall_responder` requires hitting MCID on ≥ 2 tests. Here we look at each test
independently — which tests show the most improvement, and does this vary by cohort?
MCID thresholds: VAS ≥2, TUG ≥3s, 5xSST ≥10% of pre-score (relative), Normal GS ≥0.05 m/s,
Fast GS ≥0.10 m/s, SPPB ≥1.

In [ ]:
rows = []
for col in IMPROV_COLS:
    threshold = MCID_THRESHOLDS.get(col)
    if threshold is None:
        # SST uses a relative threshold: ≥10% of pre-score
        sub_valid = df_followup[df_followup[col].notna() & df_followup["pre_5xsst_s"].notna()]
        hits = (sub_valid[col] >= sub_valid["pre_5xsst_s"] * 0.10).sum()
        sub  = sub_valid
    else:
        sub  = df_followup[df_followup[col].notna()]
        hits = (sub[col] >= threshold).sum()
    rows.append({
        "test":              IMPROV_LABELS.get(col, col),
        "n_paired":          len(sub),
        "n_mcid_hit":        int(hits),
        "mcid_hit_rate_pct": round(hits / len(sub) * 100, 1) if len(sub) > 0 else float("nan"),
        "mean_improvement":  round(sub[col].mean(), 3),
        "std_improvement":   round(sub[col].std(), 3),
    })

test_df = pd.DataFrame(rows).sort_values("mcid_hit_rate_pct", ascending=False)
test_df.to_csv(ARTEFACTS / "per_test_responder_rates.csv", index=False)
display(test_df)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = [PALETTE[i % len(PALETTE)] for i in range(len(test_df))]
bars = ax.bar(test_df["test"], test_df["mcid_hit_rate_pct"], color=colors)
ax.axhline(
    test_df["mcid_hit_rate_pct"].mean(), color="black", linestyle="--",
    label=f"Mean ({test_df['mcid_hit_rate_pct'].mean():.0f}%)"
)
for bar, n in zip(bars, test_df["n_paired"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"n={n}", ha="center", va="bottom", fontsize=8)
ax.set_title("Per-Test MCID Hit Rate (% of patients with paired data)")
ax.set_ylabel("MCID Hit Rate (%)")
ax.set_xlabel("Test")
ax.tick_params(axis="x", rotation=20)
ax.legend()
plt.tight_layout()
save_fig("per_test_bar.png")


In [ ]:
import numpy as np

cohorts  = sorted(df_followup["cohort"].dropna().unique())
hm_vals  = []
hm_annot = []
for col in IMPROV_COLS:
    threshold = MCID_THRESHOLDS.get(col)
    row, annot_row = [], []
    for cohort in cohorts:
        sub = df_followup[(df_followup["cohort"] == cohort) & df_followup[col].notna()]
        if threshold is None:
            # SST relative threshold
            sub = sub[sub["pre_5xsst_s"].notna()]
            rate_val = (sub[col] >= sub["pre_5xsst_s"] * 0.10).sum() / len(sub) * 100 if len(sub) >= 5 else float("nan")
        else:
            rate_val = (sub[col] >= threshold).sum() / len(sub) * 100 if len(sub) >= 5 else float("nan")
        if pd.isna(rate_val):
            row.append(float("nan"))
            annot_row.append(f"n={len(sub)}")
        else:
            row.append(round(rate_val, 1))
            annot_row.append(f"{rate_val:.0f}%\nn={len(sub)}")
    hm_vals.append(row)
    hm_annot.append(annot_row)

hm_df = pd.DataFrame(hm_vals,
                     index=[IMPROV_LABELS.get(c, c) for c in IMPROV_COLS],
                     columns=cohorts)

fig, ax = plt.subplots(figsize=(max(10, len(cohorts) * 1.6), 5))
sns.heatmap(hm_df, annot=pd.DataFrame(hm_annot, index=hm_df.index, columns=cohorts),
            fmt="", cmap="RdYlGn", ax=ax, vmin=0, vmax=100,
            mask=hm_df.isna(), linewidths=0.5,
            cbar_kws={"label": "MCID Hit Rate (%)"})
ax.set_title("Per-Test MCID Hit Rate (%) by Cohort  (grey = n < 5)")
ax.set_xlabel("Cohort")
ax.set_ylabel("Test")
plt.tight_layout()
save_fig("per_test_heatmap.png")


---
## Section 5 — Baseline Severity vs Improvement

Do patients with worse baseline scores improve more? A strong negative Spearman r
means sicker patients improve more — a regression-to-the-mean effect that weakens
predictive signal. We check this for each of the 6 tests.

In [ ]:
import numpy as np

PAIRS = [
    ("pre_vas",          "vas_improvement",       "VAS Pain"),
    ("pre_tug_s",        "tug_improvement",       "TUG (s)"),
    ("pre_5xsst_s",      "sst_improvement",       "5xSST (s)"),
    ("pre_normal_gs_ms", "normal_gs_improvement", "Normal GS (m/s)"),
    ("pre_fast_gs_ms",   "fast_gs_improvement",   "Fast GS (m/s)"),
    ("baseline_sppb",    "sppb_improvement",      "SPPB"),
]
valid_pairs = [(pre, imp, lbl) for pre, imp, lbl in PAIRS
               if pre in df.columns and imp in df.columns]

cols_n = 3
rows_n = (len(valid_pairs) + cols_n - 1) // cols_n
fig, axes = plt.subplots(rows_n, cols_n, figsize=(cols_n * 5, rows_n * 4))
axes = axes.flatten() if rows_n > 1 else [axes] if cols_n == 1 else list(axes)

from matplotlib.lines import Line2D
for ax, (pre_col, imp_col, label) in zip(axes, valid_pairs):
    sub = df_followup[[pre_col, imp_col, "overall_responder"]].dropna()
    colors = sub["overall_responder"].map({1: "#4C72B0", 0: "#DD8452"})
    ax.scatter(sub[pre_col], sub[imp_col], c=colors, alpha=0.45, s=22)
    if len(sub) >= 3:
        slope, intercept, *_ = stats.linregress(sub[pre_col], sub[imp_col])
        x_range = np.linspace(sub[pre_col].min(), sub[pre_col].max(), 100)
        ax.plot(x_range, slope * x_range + intercept, color="black", linewidth=1.5)
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    r, p = stats.spearmanr(sub[pre_col], sub[imp_col])
    ax.set_title(f"{label}\nSpearman r={r:.2f}, p={p:.3f}")
    ax.set_xlabel(f"Baseline ({pre_col})")
    ax.set_ylabel("Improvement")

for ax in axes[len(valid_pairs):]:
    ax.set_visible(False)

legend_handles = [
    Line2D([0],[0],marker="o",color="w",markerfacecolor="#4C72B0",markersize=8,label="Responder"),
    Line2D([0],[0],marker="o",color="w",markerfacecolor="#DD8452",markersize=8,label="Non-responder"),
]
fig.legend(handles=legend_handles, loc="lower right", fontsize=10)
plt.suptitle("Baseline Severity vs Improvement (negative r = regression to mean)", fontsize=13, y=1.01)
plt.tight_layout()
save_fig("baseline_vs_improvement.png")


**Interpreting the plots:**

- **Negative Spearman r (regression to mean):** If r < 0, patients with worse baseline scores improve more on average. This is clinically expected for pain (VAS) and gait speed — severely limited patients have more room to improve. A strong regression-to-mean effect (r < −0.3) means the baseline score is partly predicting improvement through ceiling/floor mechanics rather than true treatment response.

- **Positive or near-zero r:** If r ≥ 0, better baseline function does not predict less improvement — the treatment lifts all patients similarly regardless of starting point. This is a stronger signal of genuine treatment efficacy.

- **Ceiling effects:** Look for clusters of low-baseline patients (left side of scatter) with near-zero improvement — patients already near maximum impairment may have limited measurable change on that test's scale.

Check the Spearman r values above to identify which tests show the strongest regression-to-mean effect. Tests with r < −0.3 should be used cautiously as isolated predictors in the model — the improvement score conflates baseline severity with treatment response.

---
## Section 6 — Dropout Analysis

65% of patients have no follow-up. Is this random (MCAR) or patterned on baseline
characteristics? We compare dropouts vs completers and check dropout rates across
cohort, gender, and usage frequency.

In [ ]:
dropouts  = df_dropout_known[df_dropout_known["is_dropout"] == 1]
completers = df_dropout_known[df_dropout_known["is_dropout"] == 0]
print(f"Dropouts: {len(dropouts)}, Completers: {len(completers)}")

# Same 8 spec features as Section 3
S6_FEATS = [c for c in ["age", "baseline_sppb", "pre_normal_gs_ms", "pre_tug_s",
                          "pre_5xsst_s", "pre_vas", "pre_fast_gs_ms", "n_flags"]
             if c in df_dropout_known.columns]

rows = []
for feat in S6_FEATS:
    a = dropouts[feat].dropna()
    b = completers[feat].dropna()
    if len(a) < 5 or len(b) < 5:
        continue
    _, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    rows.append({
        "feature":            feat,
        "dropout_mean":       round(a.mean(), 2),
        "dropout_std":        round(a.std(), 2),
        "completer_mean":     round(b.mean(), 2),
        "completer_std":      round(b.std(), 2),
        "p_value":            round(p, 4),
        "significant_p05":    p < 0.05,
    })

drop_comp_df = pd.DataFrame(rows).sort_values("p_value")
drop_comp_df.to_csv(ARTEFACTS / "dropout_profile_comparison.csv", index=False)
display(drop_comp_df)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
overall_dropout_rate = df_dropout_known["is_dropout"].mean() * 100

for ax, (col, title) in zip(axes, [
    ("cohort",           "Dropout Rate by Cohort"),
    ("age_band",         "Dropout Rate by Age Band"),
    ("gender",           "Dropout Rate by Gender"),
    ("usage_frequency",  "Dropout Rate by Usage Frequency"),
]):
    vals = sorted([v for v in df_dropout_known[col].dropna().unique() if v != "__missing__"])
    rates = [(str(v), df_dropout_known[df_dropout_known[col]==v]["is_dropout"].mean()*100)
             for v in vals]
    rates.sort(key=lambda x: x[1], reverse=True)
    bar_colors = [PALETTE[i % len(PALETTE)] for i in range(len(rates))]
    ax.bar([r[0] for r in rates], [r[1] for r in rates], color=bar_colors)
    ax.axhline(overall_dropout_rate, color="red", linestyle="--", label=f"Avg {overall_dropout_rate:.0f}%")
    ax.set_title(title)
    ax.set_ylabel("Dropout Rate (%)")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=8)

plt.tight_layout()
save_fig("dropout_analysis.png")


In [ ]:
# Missingness by dropout status — does missing data predict dropout?
miss_rows = []
for feat in S6_FEATS:
    do_miss = dropouts[feat].isna().mean() * 100
    co_miss = completers[feat].isna().mean() * 100
    miss_rows.append({"feature": feat,
                      "dropout_missing_pct": round(do_miss, 1),
                      "completer_missing_pct": round(co_miss, 1),
                      "difference_pct": round(do_miss - co_miss, 1)})
miss_df = pd.DataFrame(miss_rows).sort_values("difference_pct", ascending=False)
display(miss_df)
print("\nPositive difference = dropouts have MORE missingness (potential MNAR — preview of Section 12)")
